In [4]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.pkl'):
            print(os.path.join(root, file))
            break  # just show first pkl file found, then stop
    if any(f.endswith('.pkl') for f in files):
        break

/kaggle/input/datasets/nikhilrajiv12/battery-degradation-dataset-xai/MATR/MATR_b2c38.pkl


In [5]:
import pickle
import os

# Find first HUST pkl file
hust_dir = '/kaggle/input/datasets/nikhilrajiv12/battery-degradation-dataset-xai/HUST'
first_file = sorted(os.listdir(hust_dir))[0]
file_path = os.path.join(hust_dir, first_file)
print(f"Loading: {file_path}")

with open(file_path, 'rb') as f:
    data = pickle.load(f)

print(f"\nType of loaded object: {type(data)}")

if isinstance(data, dict):
    print(f"Top-level keys (first 5): {list(data.keys())[:5]}")
    first_key = list(data.keys())[0]
    print(f"\nFirst key: {first_key}")
    print(f"Type of first value: {type(data[first_key])}")
    val = data[first_key]
    if hasattr(val, 'columns'):
        print(f"Columns: {val.columns.tolist()}")
        print(f"Shape: {val.shape}")
        print(f"\nFirst 5 rows:")
        print(val.head())
elif hasattr(data, 'columns'):
    print(f"Columns: {data.columns.tolist()}")
    print(f"Shape: {data.shape}")
    print(data.head())
else:
    print(f"Content preview: {str(data)[:500]}")

Loading: /kaggle/input/datasets/nikhilrajiv12/battery-degradation-dataset-xai/HUST/HUST_1-1.pkl

Type of loaded object: <class 'dict'>
Top-level keys (first 5): ['cell_id', 'cycle_data', 'form_factor', 'anode_material', 'cathode_material']

First key: cell_id
Type of first value: <class 'str'>


In [6]:
import pickle
import os

hust_dir = '/kaggle/input/datasets/nikhilrajiv12/battery-degradation-dataset-xai/HUST'
file_path = os.path.join(hust_dir, 'HUST_1-1.pkl')

with open(file_path, 'rb') as f:
    data = pickle.load(f)

# Check all top-level keys
print("All top-level keys:", list(data.keys()))
print(f"cell_id: {data['cell_id']}")
print(f"form_factor: {data.get('form_factor', 'N/A')}")
print(f"anode_material: {data.get('anode_material', 'N/A')}")
print(f"cathode_material: {data.get('cathode_material', 'N/A')}")

# Inspect cycle_data
cycle_data = data['cycle_data']
print(f"\nType of cycle_data: {type(cycle_data)}")

if isinstance(cycle_data, dict):
    print(f"Number of cycles: {len(cycle_data)}")
    print(f"First 5 cycle keys: {list(cycle_data.keys())[:5]}")
    first_cycle_key = list(cycle_data.keys())[0]
    first_cycle = cycle_data[first_cycle_key]
    print(f"\nType of first cycle: {type(first_cycle)}")
    if hasattr(first_cycle, 'columns'):
        print(f"Columns: {first_cycle.columns.tolist()}")
        print(f"Shape: {first_cycle.shape}")
        print(f"\nFirst 5 rows:")
        print(first_cycle.head())
elif hasattr(cycle_data, 'columns'):
    print(f"Columns: {cycle_data.columns.tolist()}")
    print(f"Shape: {cycle_data.shape}")
    print(cycle_data.head())
else:
    print(f"cycle_data preview: {str(cycle_data)[:500]}")

All top-level keys: ['cell_id', 'cycle_data', 'form_factor', 'anode_material', 'cathode_material', 'electrolyte_material', 'nominal_capacity_in_Ah', 'depth_of_charge', 'depth_of_discharge', 'already_spent_cycles', 'max_voltage_limit_in_V', 'min_voltage_limit_in_V', 'max_current_limit_in_A', 'min_current_limit_in_A', 'reference', 'description', 'charge_protocol', 'discharge_protocol']
cell_id: HUST_1-1
form_factor: cylindrical_18650
anode_material: graphite
cathode_material: LFP

Type of cycle_data: <class 'list'>
cycle_data preview: [{'cycle_number': 1, 'current_in_A': [5.49813, 5.4985100000000005, 5.4985100000000005, 5.4985100000000005, 5.49813, 5.49813, 5.49813, 5.49813, 5.49813, 5.4985100000000005, 5.49813, 5.49813, 5.49776, 5.49813, 5.4985100000000005, 5.49813, 5.49813, 5.4985100000000005, 5.49776, 5.49813, 5.4985100000000005, 5.49776, 5.4985100000000005, 5.4985100000000005, 5.4985100000000005, 5.49813, 5.4985100000000005, 5.4985100000000005, 5.49776, 5.49776, 5.49776, 5.49813, 5.4

In [7]:
# Inspect one full cycle to see all available keys and data
cycle_0 = data['cycle_data'][0]

print("Keys in one cycle dict:")
for key in cycle_0.keys():
    val = cycle_0[key]
    if isinstance(val, list):
        print(f"  {key}: list of {len(val)} values, first value = {val[0]}")
    else:
        print(f"  {key}: {val}")

print(f"\nNominal capacity: {data['nominal_capacity_in_Ah']} Ah")
print(f"Total cycles in this file: {len(data['cycle_data'])}")

Keys in one cycle dict:
  cycle_number: 1
  current_in_A: list of 836 values, first value = 5.49813
  voltage_in_V: list of 836 values, first value = 2.7731
  charge_capacity_in_Ah: list of 836 values, first value = 0.0
  discharge_capacity_in_Ah: list of 836 values, first value = 0.0
  time_in_s: list of 836 values, first value = 0
  temperature_in_C: None
  internal_resistance_in_ohm: None

Nominal capacity: 1.1 Ah
Total cycles in this file: 1504


In [8]:
# CELL 2 — Extract per-cycle features from all 77 HUST pkl files
import pickle
import numpy as np
import pandas as pd
import os

HUST_DIR      = '/kaggle/input/datasets/nikhilrajiv12/battery-degradation-dataset-xai/HUST'
OUTPUT_CSV    = '/kaggle/working/hust_features_raw.csv'
NOMINAL_CAP   = 1.1    # Ah — same for all HUST cells
TEMP_CONSTANT = 30.0   # degrees C — all HUST cells tested at 30°C

def extract_cycle_features(cycle_dict, battery_id):
    """
    Extract per-cycle scalar features from one HUST cycle dictionary.
    Returns a dict of features or None if the cycle is unusable.
    """
    cycle_num    = cycle_dict['cycle_number']
    current_arr  = np.array(cycle_dict['current_in_A'])
    voltage_arr  = np.array(cycle_dict['voltage_in_V'])
    dischg_cap   = np.array(cycle_dict['discharge_capacity_in_Ah'])
    time_arr     = np.array(cycle_dict['time_in_s'])

    # Separate discharge steps: current is negative during discharge
    discharge_mask = current_arr < -0.01   # small threshold to avoid resting steps

    if discharge_mask.sum() < 10:
        return None   # not enough discharge data points — skip cycle

    v_dis  = voltage_arr[discharge_mask]
    i_dis  = np.abs(current_arr[discharge_mask])   # use magnitude
    t_dis  = time_arr[discharge_mask]
    dc_dis = dischg_cap[discharge_mask]

    # Capacity = max discharge capacity reached in this cycle
    capacity = dc_dis.max()
    if capacity < 0.1:
        return None   # anomalous cycle — skip

    # Duration in seconds
    duration = float(t_dis[-1] - t_dis[0]) if len(t_dis) > 1 else 0.0
    if duration < 10:
        return None   # too short — skip

    return {
        'battery_id':          battery_id,
        'cycle_number':        cycle_num,
        'duration':            duration,
        'v_min':               float(v_dis.min()),
        'v_max':               float(v_dis.max()),
        'v_mean':              float(v_dis.mean()),
        'v_std':               float(v_dis.std()),
        'i_mean':              float(i_dis.mean()),
        'i_std':               float(i_dis.std()),
        't_max':               TEMP_CONSTANT,
        't_mean':              TEMP_CONSTANT,
        't_std':               0.0,
        'v_drop_rate':         float((v_dis[0] - v_dis[-1]) / duration) if duration > 0 else 0.0,
        'ambient_temperature': TEMP_CONSTANT,
        'Capacity':            float(capacity),
        'dataset':             'HUST',
    }

# --- Main extraction loop ---
all_features = []
pkl_files    = sorted([f for f in os.listdir(HUST_DIR) if f.endswith('.pkl')])

print(f"Found {len(pkl_files)} HUST pkl files.")
print(f"Starting extraction...\n")

for file_idx, fname in enumerate(pkl_files):
    fpath = os.path.join(HUST_DIR, fname)

    try:
        with open(fpath, 'rb') as f:
            data = pickle.load(f)
    except Exception as e:
        print(f"  ERROR loading {fname}: {e}")
        continue

    battery_id  = data['cell_id']
    cycle_list  = data['cycle_data']
    n_cycles    = len(cycle_list)

    # Stride logic: every 5th cycle for first 80% of life,
    # every cycle for last 20% — preserves EOL resolution
    eighty_pct  = int(n_cycles * 0.80)
    cycles_early = cycle_list[:eighty_pct]
    cycles_late  = cycle_list[eighty_pct:]

    selected_cycles = cycles_early[::5] + cycles_late   # stride=5 early, all late

    battery_features = []
    for cycle_dict in selected_cycles:
        features = extract_cycle_features(cycle_dict, battery_id)
        if features is not None:
            battery_features.append(features)

    all_features.extend(battery_features)

    print(f"  [{file_idx+1:>3}/{len(pkl_files)}] {fname:<20} "
          f"total_cycles={n_cycles:<6} "
          f"selected={len(cycles_early[::5]) + len(cycles_late):<6} "
          f"extracted={len(battery_features)}")

# --- Save to CSV ---
df_hust = pd.DataFrame(all_features)
df_hust.to_csv(OUTPUT_CSV, index=False)

print(f"\nExtraction complete.")
print(f"Total records extracted: {len(df_hust)}")
print(f"Batteries: {df_hust['battery_id'].nunique()}")
print(f"Saved to: {OUTPUT_CSV}")
print(f"\nColumn check:")
print(df_hust.dtypes)
print(f"\nCapacity range: {df_hust['Capacity'].min():.4f} – {df_hust['Capacity'].max():.4f} Ah")
print(f"Sample (first 5 rows):")
print(df_hust.head())

Found 77 HUST pkl files.
Starting extraction...

  [  1/77] HUST_1-1.pkl         total_cycles=1504   selected=542    extracted=542
  [  2/77] HUST_1-2.pkl         total_cycles=2678   selected=965    extracted=965
  [  3/77] HUST_1-3.pkl         total_cycles=1858   selected=670    extracted=670
  [  4/77] HUST_1-4.pkl         total_cycles=1500   selected=540    extracted=540
  [  5/77] HUST_1-5.pkl         total_cycles=1971   selected=711    extracted=711
  [  6/77] HUST_1-6.pkl         total_cycles=1143   selected=412    extracted=412
  [  7/77] HUST_1-7.pkl         total_cycles=1678   selected=605    extracted=605
  [  8/77] HUST_1-8.pkl         total_cycles=2285   selected=823    extracted=823
  [  9/77] HUST_10-1.pkl        total_cycles=1702   selected=614    extracted=614
  [ 10/77] HUST_10-2.pkl        total_cycles=1697   selected=612    extracted=612
  [ 11/77] HUST_10-3.pkl        total_cycles=1848   selected=666    extracted=666
  [ 12/77] HUST_10-4.pkl        total_cycles=1811

In [9]:
# CELL 3 — Compute SOH, clean anomalies, add dataset column
import pandas as pd
import numpy as np

df = pd.read_csv('/kaggle/working/hust_features_raw.csv')

print(f"Raw records: {len(df)}")
print(f"Batteries: {df['battery_id'].nunique()}")

# --- Compute SOH per battery ---
# SOH = capacity / max_capacity per battery
# max_capacity = highest capacity that battery ever recorded (early cycles)
df = df.sort_values(['battery_id', 'cycle_number']).reset_index(drop=True)

max_cap_per_bat = df.groupby('battery_id')['Capacity'].max()
df['max_capacity'] = df['battery_id'].map(max_cap_per_bat)
df['SOH'] = df['Capacity'] / df['max_capacity']
df['SOH'] = df['SOH'].clip(upper=1.0)

print(f"\nSOH range before cleaning: {df['SOH'].min():.4f} – {df['SOH'].max():.4f}")

# --- Clean anomalous cycles ---
# Remove cycles where capacity is unrealistically low
df = df[df['Capacity'] >= 0.5]

# Remove sudden SOH drops > 25% in one cycle (same threshold as NASA pipeline)
df = df.sort_values(['battery_id', 'cycle_number'])
df['SOH_prev'] = df.groupby('battery_id')['SOH'].shift(1)
df['SOH_drop'] = df['SOH_prev'] - df['SOH']
df = df[~((df['SOH_drop'] > 0.25) & (df['SOH_prev'].notna()))]
df = df.drop(columns=['SOH_prev', 'SOH_drop'])

print(f"Records after cleaning: {len(df)}")
print(f"Batteries remaining: {df['battery_id'].nunique()}")
print(f"SOH range after cleaning: {df['SOH'].min():.4f} – {df['SOH'].max():.4f}")

# --- Per battery summary ---
print(f"\nPer battery summary (first 10):")
summary = df.groupby('battery_id').agg(
    cycles       = ('cycle_number', 'count'),
    max_cycle    = ('cycle_number', 'max'),
    min_SOH      = ('SOH', 'min'),
    final_SOH    = ('SOH', lambda x: x.iloc[-1]),
    reached_EOL  = ('SOH', lambda x: (x <= 0.80).any())
).reset_index()

print(summary.head(10).to_string(index=False))
print(f"\nTotal batteries reaching EOL (SOH <= 0.80): {summary['reached_EOL'].sum()}")
print(f"Total batteries NOT reaching EOL: {(~summary['reached_EOL']).sum()}")

# --- Save ---
df.to_csv('/kaggle/working/hust_features_clean.csv', index=False)
print(f"\nSaved to /kaggle/working/hust_features_clean.csv")

Raw records: 52658
Batteries: 77

SOH range before cleaning: 0.7180 – 1.0000
Records after cleaning: 52658
Batteries remaining: 77
SOH range after cleaning: 0.7180 – 1.0000

Per battery summary (first 10):
battery_id  cycles  max_cycle  min_SOH  final_SOH  reached_EOL
  HUST_1-1     542       1504 0.752627   0.752627         True
  HUST_1-2     965       2678 0.733370   0.733379         True
  HUST_1-3     670       1858 0.744939   0.744944         True
  HUST_1-4     540       1500 0.745550   0.745550         True
  HUST_1-5     711       1971 0.749211   0.749213         True
  HUST_1-6     412       1143 0.734889   0.734891         True
  HUST_1-7     605       1678 0.746016   0.746018         True
  HUST_1-8     823       2285 0.742863   0.742863         True
 HUST_10-1     614       1702 0.753443   0.753443         True
 HUST_10-2     612       1697 0.737505   0.737505         True

Total batteries reaching EOL (SOH <= 0.80): 77
Total batteries NOT reaching EOL: 0

Saved to /kaggle

In [10]:
# CELL 4 — Compute RUL labels and temporal features for HUST
import pandas as pd
import numpy as np

df = pd.read_csv('/kaggle/working/hust_features_clean.csv')
EOL_THRESHOLD = 0.80

# --- Compute RUL labels ---
all_records = []

for bat in sorted(df['battery_id'].unique()):
    bat_df = df[df['battery_id'] == bat].sort_values('cycle_number').copy()

    eol_rows = bat_df[bat_df['SOH'] <= EOL_THRESHOLD]
    if len(eol_rows) > 0:
        eol_cycle   = eol_rows['cycle_number'].iloc[0]
        reached_eol = True
    else:
        eol_cycle   = bat_df['cycle_number'].max()
        reached_eol = False

    bat_df['eol_cycle']   = eol_cycle
    bat_df['reached_eol'] = reached_eol
    bat_df['RUL']         = (eol_cycle - bat_df['cycle_number']).clip(lower=0)

    all_records.append(bat_df)

df = pd.concat(all_records, ignore_index=True)

print(f"RUL range: {df['RUL'].min()} – {df['RUL'].max()}")
print(f"Batteries with RUL labels: {df['battery_id'].nunique()}")

# --- Add temporal features (same as NASA Cell 15) ---
df = df.sort_values(['battery_id', 'cycle_number'])

for col in ['duration', 'v_mean', 't_mean', 'i_mean']:
    df[f'{col}_roll5'] = (df.groupby('battery_id')[col]
                           .transform(lambda x: x.rolling(5, min_periods=1).mean()))
    df[f'{col}_trend'] = df[col] - df[f'{col}_roll5']

df['SOH_prev']   = df.groupby('battery_id')['SOH'].shift(1)
df['SOH_change'] = df['SOH'] - df['SOH_prev']
df['SOH_roll5']  = (df.groupby('battery_id')['SOH']
                     .transform(lambda x: x.rolling(5, min_periods=1).mean()))

df = df.dropna(subset=['SOH_prev']).reset_index(drop=True)

print(f"Records after adding temporal features: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# --- Normalized RUL (0 to 1) ---
# This is Step 2 of our plan — normalized RUL removes scale difference
# between batteries with different lifespans (1100 vs 2700 cycles)
max_rul_per_bat   = df.groupby('battery_id')['RUL'].transform('max')
df['RUL_normalized'] = df['RUL'] / max_rul_per_bat.replace(0, 1)
df['RUL_normalized'] = df['RUL_normalized'].clip(0, 1)

print(f"\nNormalized RUL range: {df['RUL_normalized'].min():.4f} – {df['RUL_normalized'].max():.4f}")

# --- Save ---
df.to_csv('/kaggle/working/hust_features_final.csv', index=False)
print(f"\nSaved to /kaggle/working/hust_features_final.csv")
print(f"Shape: {df.shape}")
print(f"\nSample:")
print(df[['battery_id', 'cycle_number', 'SOH', 'RUL', 'RUL_normalized']].head(10).to_string(index=False))

RUL range: 0 – 2322
Batteries with RUL labels: 77
Records after adding temporal features: 52581
Columns: ['battery_id', 'cycle_number', 'duration', 'v_min', 'v_max', 'v_mean', 'v_std', 'i_mean', 'i_std', 't_max', 't_mean', 't_std', 'v_drop_rate', 'ambient_temperature', 'Capacity', 'dataset', 'max_capacity', 'SOH', 'eol_cycle', 'reached_eol', 'RUL', 'duration_roll5', 'duration_trend', 'v_mean_roll5', 'v_mean_trend', 't_mean_roll5', 't_mean_trend', 'i_mean_roll5', 'i_mean_trend', 'SOH_prev', 'SOH_change', 'SOH_roll5']

Normalized RUL range: 0.0000 – 1.0000

Saved to /kaggle/working/hust_features_final.csv
Shape: (52581, 33)

Sample:
battery_id  cycle_number      SOH  RUL  RUL_normalized
  HUST_1-1             6 0.997477 1348        1.000000
  HUST_1-1            11 0.996411 1343        0.996291
  HUST_1-1            16 0.995378 1338        0.992582
  HUST_1-1            21 0.994365 1333        0.988872
  HUST_1-1            26 0.994286 1328        0.985163
  HUST_1-1            31 0.9932

In [11]:
# CELL 5 — Train RUL model on HUST dataset (normalized RUL target)
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/kaggle/working/hust_features_final.csv')

RUL_FEATURE_COLS = [
    'duration', 'v_min', 'v_max', 'v_mean', 'v_std',
    'i_mean', 'i_std', 't_max', 't_mean', 't_std',
    'v_drop_rate', 'ambient_temperature', 'cycle_number', 'SOH',
    'duration_roll5', 'duration_trend',
    'v_mean_roll5',   'v_mean_trend',
    't_mean_roll5',   't_mean_trend',
    'i_mean_roll5',   'i_mean_trend',
    'SOH_prev', 'SOH_change', 'SOH_roll5'
]

# --- Leave-One-Battery-Out cross validation ---
# Too expensive to do all 77 — use 10 random test batteries
# Train on remaining 67, test on held-out 10
np.random.seed(42)
all_batteries  = sorted(df['battery_id'].unique())
test_batteries = list(np.random.choice(all_batteries, size=10, replace=False))
train_batteries = [b for b in all_batteries if b not in test_batteries]

print(f"Training batteries: {len(train_batteries)}")
print(f"Test batteries: {len(test_batteries)}")

train_df = df[df['battery_id'].isin(train_batteries)]
test_df  = df[df['battery_id'].isin(test_batteries)]

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

X_train = train_df[RUL_FEATURE_COLS]
y_train = train_df['RUL_normalized']
X_test  = test_df[RUL_FEATURE_COLS]
y_test  = test_df['RUL_normalized']

sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s  = sc.transform(X_test)

print("\nTraining Random Forest on HUST (normalized RUL)...")
rf_rul = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_rul.fit(X_train_s, y_train)
pred_normalized = rf_rul.predict(X_test_s)
pred_normalized = np.clip(pred_normalized, 0, 1)

# --- Metrics on normalized RUL ---
mae_norm  = mean_absolute_error(y_test, pred_normalized)
rmse_norm = np.sqrt(mean_squared_error(y_test, pred_normalized))
r2_norm   = r2_score(y_test, pred_normalized)

print(f"\nHUST-only RUL Model (Normalized RUL target):")
print(f"  MAE  (normalized): {mae_norm:.4f}")
print(f"  RMSE (normalized): {rmse_norm:.4f}")
print(f"  R²   (normalized): {r2_norm:.4f}")

# --- Convert back to cycle counts for interpretability ---
max_rul_per_bat = test_df.groupby('battery_id')['RUL'].max()
test_df = test_df.copy()
test_df['pred_normalized'] = pred_normalized
test_df['max_rul']         = test_df['battery_id'].map(max_rul_per_bat)
test_df['pred_RUL']        = test_df['pred_normalized'] * test_df['max_rul']
test_df['pred_RUL']        = test_df['pred_RUL'].clip(lower=0)

mae_cycles  = mean_absolute_error(test_df['RUL'], test_df['pred_RUL'])
rmse_cycles = np.sqrt(mean_squared_error(test_df['RUL'], test_df['pred_RUL']))
r2_cycles   = r2_score(test_df['RUL'], test_df['pred_RUL'])

print(f"\nHUST-only RUL Model (converted back to cycles):")
print(f"  MAE  (cycles): {mae_cycles:.1f}")
print(f"  RMSE (cycles): {rmse_cycles:.1f}")
print(f"  R²   (cycles): {r2_cycles:.4f}")

# --- Per battery breakdown ---
print(f"\nPer test battery results:")
print(f"{'Battery':<15} {'MAE (cyc)':>12} {'R²':>8} {'Samples':>10}")
print("-" * 48)
for bat in sorted(test_batteries):
    b     = test_df[test_df['battery_id'] == bat]
    b_mae = mean_absolute_error(b['RUL'], b['pred_RUL'])
    b_r2  = r2_score(b['RUL'], b['pred_RUL'])
    print(f"{bat:<15} {b_mae:>12.1f} {b_r2:>8.4f} {len(b):>10}")

# --- Feature importance ---
importances = pd.Series(rf_rul.feature_importances_, index=RUL_FEATURE_COLS)
importances = importances.sort_values(ascending=False)
print(f"\nTop 10 most important features:")
print(importances.head(10).to_string())

# --- Save model and scaler ---
import pickle
with open('/kaggle/working/rf_rul_hust.pkl', 'wb') as f:
    pickle.dump(rf_rul, f)
with open('/kaggle/working/scaler_rul_hust.pkl', 'wb') as f:
    pickle.dump(sc, f)

print(f"\nModel saved to /kaggle/working/rf_rul_hust.pkl")

Training batteries: 67
Test batteries: 10
Training samples: 45755
Test samples: 6826

Training Random Forest on HUST (normalized RUL)...

HUST-only RUL Model (Normalized RUL target):
  MAE  (normalized): 0.0100
  RMSE (normalized): 0.0212
  R²   (normalized): 0.9955

HUST-only RUL Model (converted back to cycles):
  MAE  (cycles): 17.2
  RMSE (cycles): 36.3
  R²   (cycles): 0.9957

Per test battery results:
Battery            MAE (cyc)       R²    Samples
------------------------------------------------
HUST_1-1                12.8   0.9956        541
HUST_1-5                48.0   0.9776        710
HUST_10-3               19.1   0.9955        665
HUST_3-6                20.4   0.9974        897
HUST_4-5                 6.1   0.9994        562
HUST_5-7                21.0   0.9938        521
HUST_6-2                 5.4   0.9996        687
HUST_6-5                16.1   0.9974        784
HUST_7-1                 4.8   0.9998        608
HUST_8-6                13.5   0.9986        851



In [12]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if 'features_final' in f.lower() or 'features' in f.lower():
            print(os.path.join(root, f))

/kaggle/input/datasets/anupkumarml/nasa-features-final/features_final (1).csv


In [13]:
# CELL 6 — Combined NASA + HUST RUL Model
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

NASA_PATH = '/kaggle/input/datasets/anupkumarml/nasa-features-final/features_final (1).csv'

df_hust = pd.read_csv('/kaggle/working/hust_features_final.csv')
df_nasa = pd.read_csv(NASA_PATH)

print(f"NASA raw records: {len(df_nasa)}, batteries: {df_nasa['battery_id'].nunique()}")
print(f"HUST records: {len(df_hust)}, batteries: {df_hust['battery_id'].nunique()}")

# --- Add RUL labels to NASA ---
EOL_THRESHOLD = 0.80
nasa_records  = []
for bat in sorted(df_nasa['battery_id'].unique()):
    bat_df   = df_nasa[df_nasa['battery_id'] == bat].sort_values('cycle_number').copy()
    eol_rows = bat_df[bat_df['SOH'] <= EOL_THRESHOLD]
    if len(eol_rows) > 0:
        eol_cycle   = eol_rows['cycle_number'].iloc[0]
        reached_eol = True
    else:
        eol_cycle   = bat_df['cycle_number'].max()
        reached_eol = False
    bat_df['eol_cycle']   = eol_cycle
    bat_df['reached_eol'] = reached_eol
    bat_df['RUL']         = (eol_cycle - bat_df['cycle_number']).clip(lower=0)
    bat_df['dataset']     = 'NASA'
    nasa_records.append(bat_df)

df_nasa_rul = pd.concat(nasa_records, ignore_index=True)

# Add temporal features to NASA if not already present
df_nasa_rul = df_nasa_rul.sort_values(['battery_id', 'cycle_number'])
for col in ['duration', 'v_mean', 't_mean', 'i_mean']:
    if f'{col}_roll5' not in df_nasa_rul.columns:
        df_nasa_rul[f'{col}_roll5'] = (df_nasa_rul.groupby('battery_id')[col]
                                        .transform(lambda x: x.rolling(5, min_periods=1).mean()))
        df_nasa_rul[f'{col}_trend'] = df_nasa_rul[col] - df_nasa_rul[f'{col}_roll5']

if 'SOH_prev' not in df_nasa_rul.columns:
    df_nasa_rul['SOH_prev']   = df_nasa_rul.groupby('battery_id')['SOH'].shift(1)
    df_nasa_rul['SOH_change'] = df_nasa_rul['SOH'] - df_nasa_rul['SOH_prev']
    df_nasa_rul['SOH_roll5']  = (df_nasa_rul.groupby('battery_id')['SOH']
                                  .transform(lambda x: x.rolling(5, min_periods=1).mean()))

df_nasa_rul = df_nasa_rul.dropna(subset=['SOH_prev'])

# Normalized RUL for NASA
max_rul_nasa = df_nasa_rul.groupby('battery_id')['RUL'].transform('max')
df_nasa_rul['RUL_normalized'] = (df_nasa_rul['RUL'] / max_rul_nasa.replace(0, 1)).clip(0, 1)

print(f"NASA records after processing: {len(df_nasa_rul)}")

# --- Combine ---
RUL_FEATURE_COLS = [
    'duration', 'v_min', 'v_max', 'v_mean', 'v_std',
    'i_mean', 'i_std', 't_max', 't_mean', 't_std',
    'v_drop_rate', 'ambient_temperature', 'cycle_number', 'SOH',
    'duration_roll5', 'duration_trend',
    'v_mean_roll5',   'v_mean_trend',
    't_mean_roll5',   't_mean_trend',
    'i_mean_roll5',   'i_mean_trend',
    'SOH_prev', 'SOH_change', 'SOH_roll5'
]

common_cols  = RUL_FEATURE_COLS + ['RUL', 'RUL_normalized', 'battery_id', 'dataset']
df_combined  = pd.concat([
    df_nasa_rul[common_cols],
    df_hust[common_cols]
], ignore_index=True)

print(f"\nCombined: {len(df_combined)} records, {df_combined['battery_id'].nunique()} batteries")
print(f"NASA: {len(df_combined[df_combined['dataset']=='NASA'])} records")
print(f"HUST: {len(df_combined[df_combined['dataset']=='HUST'])} records")

# --- Train/test split ---
# Hold out same NASA test batteries as before + 5 HUST batteries
nasa_test_bats = ['B0018', 'B0032']
hust_test_bats = list(np.random.RandomState(42).choice(
    sorted(df_hust['battery_id'].unique()), size=5, replace=False))

test_bats  = nasa_test_bats + hust_test_bats
train_bats = [b for b in df_combined['battery_id'].unique() if b not in test_bats]

train_df = df_combined[df_combined['battery_id'].isin(train_bats)]
test_df  = df_combined[df_combined['battery_id'].isin(test_bats)]

print(f"\nTrain: {len(train_df)} records, {len(train_bats)} batteries")
print(f"Test:  {len(test_df)} records, {len(test_bats)} batteries")
print(f"Test batteries: {test_bats}")

X_train = train_df[RUL_FEATURE_COLS]
y_train = train_df['RUL_normalized']
X_test  = test_df[RUL_FEATURE_COLS]
y_test  = test_df['RUL_normalized']

sc_combined = StandardScaler()
X_train_s   = sc_combined.fit_transform(X_train)
X_test_s    = sc_combined.transform(X_test)

print("\nTraining combined NASA + HUST RUL model...")
rf_combined = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_combined.fit(X_train_s, y_train)
pred_norm = rf_combined.predict(X_test_s)
pred_norm = np.clip(pred_norm, 0, 1)

mae_norm = mean_absolute_error(y_test, pred_norm)
r2_norm  = r2_score(y_test, pred_norm)

print(f"\nCombined Model — Normalized RUL:")
print(f"  MAE (normalized): {mae_norm:.4f}")
print(f"  R²  (normalized): {r2_norm:.4f}")

# Per battery results
test_df = test_df.copy()
test_df['pred_norm'] = pred_norm
max_rul_test = test_df.groupby('battery_id')['RUL'].transform('max')
test_df['pred_RUL'] = (test_df['pred_norm'] * max_rul_test).clip(lower=0)

print(f"\nPer battery breakdown:")
print(f"{'Battery':<15} {'Dataset':>8} {'MAE (cyc)':>12} {'R²':>8}")
print("-" * 48)
for bat in sorted(test_bats):
    b     = test_df[test_df['battery_id'] == bat]
    ds    = b['dataset'].iloc[0]
    b_mae = mean_absolute_error(b['RUL'], b['pred_RUL'])
    b_r2  = r2_score(b['RUL'], b['pred_RUL'])
    print(f"{bat:<15} {ds:>8} {b_mae:>12.1f} {b_r2:>8.4f}")

# Save everything
with open('/kaggle/working/rf_rul_combined.pkl', 'wb') as f:
    pickle.dump(rf_combined, f)
with open('/kaggle/working/scaler_rul_combined.pkl', 'wb') as f:
    pickle.dump(sc_combined, f)
with open('/kaggle/working/rul_feature_cols.pkl', 'wb') as f:
    pickle.dump(RUL_FEATURE_COLS, f)

print(f"\nAll models saved to /kaggle/working/")

NASA raw records: 1615, batteries: 18
HUST records: 52581, batteries: 77
NASA records after processing: 1597

Combined: 54178 records, 95 batteries
NASA: 1597 records
HUST: 52581 records

Train: 51009 records, 88 batteries
Test:  3169 records, 7 batteries
Test batteries: ['B0018', 'B0032', np.str_('HUST_1-5'), np.str_('HUST_4-5'), np.str_('HUST_10-3'), np.str_('HUST_1-1'), np.str_('HUST_5-7')]

Training combined NASA + HUST RUL model...

Combined Model — Normalized RUL:
  MAE (normalized): 0.0178
  R²  (normalized): 0.9858

Per battery breakdown:
Battery          Dataset    MAE (cyc)       R²
------------------------------------------------
B0018               NASA          4.8   0.8846
B0032               NASA          3.1   0.9076
HUST_1-1            HUST          9.5   0.9978
HUST_1-5            HUST         49.2   0.9776
HUST_10-3           HUST         22.1   0.9937
HUST_4-5            HUST          6.4   0.9993
HUST_5-7            HUST         22.8   0.9927

All models saved to /